# 03 · Preprocesamiento y reducción de dimensionalidad — Titanic

**Objetivo (Fase 3):** preparar los datos para un futuro modelado aplicando las decisiones tomadas en el EDA:

1. Limpieza: eliminar columnas redundantes y con fuga de información
2. Ingeniería de variables: indicadores de faltantes, tamaño de familia
3. Tratamiento de faltantes (imputación)
4. Transformación de variables sesgadas
5. Codificación de variables categóricas
6. Escalado de variables numéricas
7. Reducción de dimensionalidad con **PCA** (y comparación con **t-SNE**)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

sns.set_theme(style="whitegrid", palette="deep")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/titanic.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Ejecutar desde el repositorio o su carpeta notebooks.")
DATA = ROOT / "data"
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
OUT = ROOT / "reports" / "mejoras"
OUT.mkdir(parents=True, exist_ok=True)

def guardar(nombre):
    plt.tight_layout(); plt.savefig(FIG / nombre, dpi=120, bbox_inches="tight"); plt.show()

df = pd.read_csv(DATA / "titanic.csv")
print("Dimensiones originales:", df.shape)

Dimensiones originales: (891, 15)


## 1. Eliminación de columnas redundantes

Según el EDA, estas columnas repiten información de otras:
- `alive` = `survived` (además, usarla para predecir sería **fuga de información**)
- `class` = `pclass`, `embark_town` = `embarked`
- `alone` y `adult_male` se derivan de otras variables (`sibsp`, `parch`, `who`)

`who` (man/woman/child) se elimina porque combina `sex` y `age`; dejarla duplicaría señal y además usa la edad antes de imputarla.

In [2]:
redundantes = ["alive", "class", "embark_town", "alone", "adult_male", "who"]
df = df.drop(columns=redundantes)
print("Dimensiones tras eliminar redundantes:", df.shape)
df.head()

Dimensiones tras eliminar redundantes: (891, 9)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,deck
0,0,3,male,22.0,1,0,7.2500,S,NaN
1,1,1,female,38.0,1,0,71.2833,C,C
2,1,3,female,26.0,0,0,7.9250,S,NaN
3,1,1,female,35.0,1,0,53.1000,S,C
4,0,3,male,35.0,0,0,8.0500,S,NaN


## 2. Ingeniería de variables (antes de imputar)

Los indicadores de faltantes se crean **antes** de imputar para no perder la información de que el dato estaba ausente.

In [3]:
df["tiene_cubierta"] = df["deck"].notna().astype(int)
df["edad_faltante"] = df["age"].isna().astype(int)
df["tamano_familia"] = df["sibsp"] + df["parch"] + 1
df = df.drop(columns="deck")   # 77 % faltante: se reemplaza por el indicador
df[["tiene_cubierta", "edad_faltante", "tamano_familia"]].describe().T

,count,mean,std,min,25%,50%,75%,max
tiene_cubierta,891.0,0.227834,0.419670,0.0,0.0,0.0,0.0,1.0
edad_faltante,891.0,0.198653,0.399210,0.0,0.0,0.0,0.0,1.0
tamano_familia,891.0,1.904602,1.613459,1.0,1.0,1.0,2.0,11.0


## 3. Imputación de valores faltantes

- **`age`:** mediana agrupada por `pclass` y `sex` (la edad típica cambia según la clase y el sexo; la mediana es robusta al sesgo).
- **`embarked`:** moda.

In [4]:
print("Faltantes antes:\n", df.isna().sum()[df.isna().sum() > 0], "\n")

medianas = df.groupby(["pclass", "sex"])["age"].median()
print("Medianas de edad usadas para imputar:\n", medianas.unstack(), "\n")
edad_antes = df["age"].copy()
df["age"] = df["age"].fillna(df.groupby(["pclass", "sex"])["age"].transform("median"))

moda_emb = df["embarked"].mode()[0]
df["embarked"] = df["embarked"].fillna(moda_emb)
print("Moda de embarked:", moda_emb)
print("Faltantes después:", df.isna().sum().sum())

Faltantes antes:
 age         177
embarked      2
dtype: int64 

Medianas de edad usadas para imputar:
 sex     female  male
pclass              
1         35.0  40.0
2         28.0  30.0
3         21.5  25.0 

Moda de embarked: S
Faltantes después: 0


In [5]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(edad_antes.dropna(), label="antes (sin faltantes)", fill=True, alpha=0.3, ax=ax)
sns.kdeplot(df["age"], label="después de imputar", fill=True, alpha=0.3, ax=ax)
ax.set_title("Distribución de la edad antes y después de la imputación"); ax.legend()
guardar("03_imputacion_edad.png")

/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58046/3030717170.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / nombre, dpi=120, bbox_inches="tight"); plt.show()


La imputación por grupos introduce valores repetidos en las medianas. El gráfico permite examinar el cambio de distribución, pero no demuestra que se recuperen las edades verdaderas ni que esta estrategia sea mejor que otras sin una comparación adicional.

## 4. Transformación de variables sesgadas

`fare` tenía asimetría ≈ 4.8. Se aplica `log1p` (log(1 + x), que admite tarifas en 0).

In [6]:
df["log_fare"] = np.log1p(df["fare"])
print(f"Asimetría fare: {df.fare.skew():.2f} → log_fare: {df.log_fare.skew():.2f}")
df = df.drop(columns="fare")

Asimetría fare: 4.79 → log_fare: 0.39


## 5. Codificación de variables categóricas

| Variable | Tipo | Codificación | Razón |
|----------|------|--------------|-------|
| `sex` | Nominal binaria | Binaria (female = 1) | Solo 2 categorías: una columna basta |
| `embarked` | Nominal (3 categorías) | **One-Hot** (`drop="first"`) | No hay orden entre puertos; se elimina una columna para evitar la trampa de variables ficticias |
| `pclass` | Ordinal | Se mantiene numérica (1, 2, 3) | El orden tiene significado (1.ª > 2.ª > 3.ª) |

In [7]:
df["sex"] = (df["sex"] == "female").astype(int)
df = df.rename(columns={"sex": "sexo_mujer"})

ohe = OneHotEncoder(drop="first", sparse_output=False)
emb = ohe.fit_transform(df[["embarked"]])
emb_df = pd.DataFrame(emb, columns=ohe.get_feature_names_out(["embarked"]), index=df.index).astype(int)
df = pd.concat([df.drop(columns="embarked"), emb_df], axis=1)
print("Columnas creadas por One-Hot:", list(emb_df.columns))
df.head()

Columnas creadas por One-Hot: ['embarked_Q', 'embarked_S']


,survived,pclass,sexo_mujer,age,sibsp,parch,tiene_cubierta,edad_faltante,tamano_familia,log_fare,embarked_Q,embarked_S
0,0,3,0,22.0,1,0,0,0,2,2.110213,0,1
1,1,1,1,38.0,1,0,1,0,2,4.280593,0,0
2,1,3,1,26.0,0,0,0,0,1,2.188856,0,1
3,1,1,1,35.0,1,0,1,0,2,3.990834,0,1
4,0,3,0,35.0,0,0,0,0,1,2.202765,0,1


## 6. Escalado

Las variables tienen escalas muy distintas (edad 0–80, `log_fare` 0–6, indicadores 0–1). PCA y muchos modelos (regresión logística, KNN, SVM) son sensibles a la escala, así que se estandariza con **StandardScaler** (media 0, desviación 1). Como comparación se muestra **MinMaxScaler** (rango 0–1).

La variable objetivo `survived` **no se escala ni entra al PCA**: se separa para usarla solo como color en las gráficas.

**Alcance exploratorio:** en este notebook se ajusta con toda la muestra para describir los datos y visualizar PCA. Para evaluar un modelo futuro, dividir los datos originales antes de ajustar imputación, codificación, escalado y PCA. La sección 9 de este notebook demuestra ese flujo.

In [8]:
y = df["survived"]
X = df.drop(columns="survived")
print("Variables de entrada:", list(X.columns))

scaler = StandardScaler()
X_std = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_mm = pd.DataFrame(MinMaxScaler().fit_transform(X), columns=X.columns)

comparacion = pd.DataFrame({
    "media_original": X.mean(), "std_original": X.std(),
    "media_standard": X_std.mean(), "std_standard": X_std.std(ddof=0),
    "min_minmax": X_mm.min(), "max_minmax": X_mm.max(),
}).round(3)
comparacion

Variables de entrada: ['pclass', 'sexo_mujer', 'age', 'sibsp', 'parch', 'tiene_cubierta', 'edad_faltante', 'tamano_familia', 'log_fare', 'embarked_Q', 'embarked_S']


,media_original,std_original,media_standard,std_standard,min_minmax,max_minmax
pclass,2.309,0.836,-0.0,1.0,0.0,1.0
sexo_mujer,0.352,0.478,0.0,1.0,0.0,1.0
age,29.112,13.304,0.0,1.0,0.0,1.0
sibsp,0.523,1.103,0.0,1.0,0.0,1.0
parch,0.382,0.806,0.0,1.0,0.0,1.0
tiene_cubierta,0.228,0.420,0.0,1.0,0.0,1.0
edad_faltante,0.199,0.399,-0.0,1.0,0.0,1.0
tamano_familia,1.905,1.613,-0.0,1.0,0.0,1.0
log_fare,2.962,0.969,-0.0,1.0,0.0,1.0
embarked_Q,0.086,0.281,0.0,1.0,0.0,1.0


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
cols = ["age", "log_fare", "tamano_familia", "sibsp", "parch"]
sns.boxplot(data=X[cols], ax=axes[0], palette="Set3"); axes[0].set_title("Antes del escalado")
sns.boxplot(data=X_std[cols], ax=axes[1], palette="Set3"); axes[1].set_title("Después de StandardScaler")
guardar("03_escalado_antes_despues.png")

/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58046/3030717170.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / nombre, dpi=120, bbox_inches="tight"); plt.show()


In [10]:
procesado = X_std.copy(); procesado["survived"] = y.values
procesado.to_csv(DATA / "titanic_procesado.csv", index=False)
print("Dataset procesado guardado:", procesado.shape)

Dataset procesado guardado: (891, 12)


## 7. Reducción de dimensionalidad con PCA

### 7.1 Varianza explicada

In [11]:
pca = PCA()
X_pca = pca.fit_transform(X_std)
var = pca.explained_variance_ratio_
acum = np.cumsum(var)
n90 = int(np.argmax(acum >= 0.90) + 1)

tabla_var = pd.DataFrame({"componente": [f"PC{i+1}" for i in range(len(var))],
                          "varianza_%": (var*100).round(2), "acumulada_%": (acum*100).round(2)})
print(f"Componentes necesarios para explicar ≥ 90 % de la varianza: {n90} de {X_std.shape[1]}")
tabla_var

Componentes necesarios para explicar ≥ 90 % de la varianza: 7 de 11


,componente,varianza_%,acumulada_%
0,PC1,26.35,26.35
1,PC2,22.76,49.11
2,PC3,15.59,64.70
3,PC4,8.71,73.41
4,PC5,6.56,79.97
5,PC6,6.30,86.27
6,PC7,4.92,91.19
7,PC8,4.11,95.30
8,PC9,2.98,98.28
9,PC10,1.72,100.00


In [12]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(range(1, len(var)+1), var*100, alpha=0.7, label="varianza individual")
ax.plot(range(1, len(var)+1), acum*100, "o-", color="red", label="varianza acumulada")
ax.axhline(90, ls="--", color="gray"); ax.axvline(n90, ls=":", color="gray")
ax.text(n90 + 0.1, 50, f"{n90} componentes\n≥ 90 %", color="gray")
ax.set_xlabel("componente principal"); ax.set_ylabel("% de varianza"); ax.set_xticks(range(1, len(var)+1))
ax.set_title("Varianza explicada por componente (scree plot)"); ax.legend()
guardar("03_pca_varianza_explicada.png")

/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58046/3030717170.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / nombre, dpi=120, bbox_inches="tight"); plt.show()


### 7.2 Cargas (loadings): ¿qué representa cada componente?

In [13]:
cargas = pd.DataFrame(pca.components_[:4].T, index=X.columns, columns=["PC1", "PC2", "PC3", "PC4"])
plt.figure(figsize=(7, 5.5))
sns.heatmap(cargas, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Cargas de las variables en los 4 primeros componentes")
guardar("03_pca_cargas.png")
cargas.round(3)

/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58046/3030717170.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / nombre, dpi=120, bbox_inches="tight"); plt.show()


,PC1,PC2,PC3,PC4
pclass,-0.267,0.502,0.085,0.087
sexo_mujer,0.227,0.012,-0.232,0.750
age,-0.019,-0.402,0.042,-0.307
sibsp,0.389,0.332,-0.011,-0.316
parch,0.412,0.246,0.023,0.078
tiene_cubierta,0.269,-0.436,-0.138,-0.049
edad_faltante,-0.157,0.149,-0.397,-0.436
tamano_familia,0.472,0.350,0.004,-0.177
log_fare,0.459,-0.239,-0.148,-0.073
embarked_Q,-0.154,0.150,-0.579,-0.004


### 7.3 Proyección de los pasajeros en PC1 y PC2

In [14]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (col, titulo, pal) in zip(axes, [(y.map({0: "No sobrevivió", 1: "Sobrevivió"}), "Color = supervivencia", "Set1"),
                                        (X["pclass"].map({1: "1.ª", 2: "2.ª", 3: "3.ª"}), "Color = clase", "viridis")]):
    sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=col, hue_order=sorted(col.unique()), palette=pal, alpha=0.6, s=30, ax=ax)
    ax.set_xlabel(f"PC1 ({var[0]*100:.1f} %)"); ax.set_ylabel(f"PC2 ({var[1]*100:.1f} %)")
    ax.set_title(f"Proyección PCA — {titulo}")

# biplot: vectores de las variables sobre el primer gráfico
esc = 3
for i, v in enumerate(X.columns):
    axes[0].arrow(0, 0, pca.components_[0, i]*esc, pca.components_[1, i]*esc, color="black", alpha=0.7, head_width=0.08)
    axes[0].text(pca.components_[0, i]*esc*1.12, pca.components_[1, i]*esc*1.12, v, fontsize=8)
guardar("03_pca_proyeccion.png")

/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58046/3030717170.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / nombre, dpi=120, bbox_inches="tight"); plt.show()


In [15]:
pc_df = pd.DataFrame(X_pca[:, :3], columns=["PC1", "PC2", "PC3"]); pc_df["survived"] = y.values
print("Media de cada componente por grupo de supervivencia:")
print(pc_df.groupby("survived").mean().round(3))
print("\nCorrelación de cada componente con survived:")
print(pc_df.corr()["survived"].drop("survived").round(3))

Media de cada componente por grupo de supervivencia:
            PC1    PC2    PC3
survived                     
0        -0.385  0.286  0.244
1         0.619 -0.459 -0.391

Correlación de cada componente con survived:
PC1    0.287
PC2   -0.229
PC3   -0.236
Name: survived, dtype: float64


### 7.4 Comparación con t-SNE (método no lineal)

PCA solo captura relaciones lineales. t-SNE es un método no lineal que preserva vecindades locales; sirve para **visualizar** agrupaciones, aunque sus ejes no son interpretables y no se usa para transformar datos nuevos.

In [16]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init="pca", learning_rate="auto")
X_tsne = tsne.fit_transform(X_std)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=y.map({0: "No sobrevivió", 1: "Sobrevivió"}),
                palette="Set1", alpha=0.6, s=25, ax=axes[0])
axes[0].set_title("t-SNE — color = supervivencia")
etiqueta = X["sexo_mujer"].map({0: "H", 1: "M"}) + "-" + X["pclass"].astype(str) + "ª"
sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=etiqueta, palette="tab10", alpha=0.6, s=25, ax=axes[1])
axes[1].set_title("t-SNE — color = sexo y clase")
for ax in axes: ax.set_xlabel("dimensión 1"); ax.set_ylabel("dimensión 2")
guardar("03_tsne.png")

/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58046/3030717170.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / nombre, dpi=120, bbox_inches="tight"); plt.show()


### 7.5 Comparación sin redundancia familiar

Se comparan dos PCA exploratorios sobre las mismas filas: las 11 variables originales y 9 variables conservando solo `tamano_familia` en lugar de incluir también `sibsp` y `parch`. Se ajusta con toda la muestra para describir su geometría, sin evaluar predicción. Eliminar la dependencia exacta no garantiza mejorar un futuro modelo.

In [17]:
comparaciones = []
for nombre, datos in [("11 variables originales", X),
                      ("9 variables sin redundancia familiar", X.drop(columns=["sibsp", "parch"]))]:
    z = StandardScaler().fit_transform(datos)
    pca_comparacion = PCA().fit(z)
    acumulada = pca_comparacion.explained_variance_ratio_.cumsum()
    comparaciones.append(dict(variante=nombre, variables=datos.shape[1],
        rango=int(np.linalg.matrix_rank(z)), componentes_90=int(np.searchsorted(acumulada, .9)+1),
        varianza_2pc=float(acumulada[1]), ultima_varianza=float(pca_comparacion.explained_variance_ratio_[-1])))
comparacion = pd.DataFrame(comparaciones)
comparacion.to_csv(OUT / "comparacion_pca.csv", index=False)
display(comparacion)

,variante,variables,rango,componentes_90,varianza_2pc,ultima_varianza
0,11 variables originales,11,10,7,0.491122,1.770906e-17
1,9 variables sin redundancia familiar,9,9,7,0.484795,2.011967e-02


**Resultado:** la representación original tiene rango 10 con 11 variables; la alternativa tiene rango 9 con 9 variables. Ambas requieren 7 componentes para conservar al menos 90 % de varianza. La alternativa elimina la dependencia exacta, pero no reduce ese número de componentes.

## 8. Conclusiones del preprocesamiento y la reducción

- **Resultado del preprocesamiento:** de 15 columnas originales (con faltantes, texto y redundancias) se pasó a **11 variables numéricas, sin faltantes y estandarizadas**, guardadas en `data/titanic_procesado.csv` junto con la variable objetivo.
- **PCA:** la varianza está bastante repartida: se necesitan **7 de 11 componentes para explicar el 90 %** (PC1 ≈ 26 %, PC2 ≈ 23 %). Esto describe la varianza de esta representación; no demuestra independencia entre las variables. El **PC11 explica 0 %** porque `tamano_familia` es una combinación lineal exacta de `sibsp + parch`: PCA detecta así la redundancia perfecta.
- **Interpretación de los componentes (según las cargas):**
  - **PC1 ≈ "familia y gasto"**: cargas altas de `tamano_familia`, `log_fare`, `parch` y `sibsp`, y negativa de `pclass`. Valores altos = familias que pagaron más.
  - **PC2 ≈ "pasajero joven de clase baja"**: positivo en `pclass` y familia, negativo en `tiene_cubierta`, `age` y `log_fare`.
  - **PC3 ≈ "puerto de embarque"**: Southampton frente a Queenstown.
  - **PC4 ≈ "sexo"**: dominado por `sexo_mujer`.
- **Separación de clases:** los sobrevivientes tienen en promedio PC1 más alto y PC2 más bajo (mayor gasto, mejor clase, cubierta registrada), pero en el plano PC1–PC2 los grupos se **superponen** porque el factor más importante (el sexo) queda principalmente en el PC4. PCA maximiza varianza, no separación de clases.
- **t-SNE** permite visualizar agrupaciones relacionadas con sexo y clase en esta configuración. No confirma hipótesis ni demuestra grupos estables; depende de escalado, semilla y perplejidad.
- **Para un futuro modelo:** comparar representaciones mediante validación, ajustando el preprocesamiento solo con entrenamiento. Las secciones 7.5 y 9 comparan la representación sin redundancia familiar y generan una partición ilustrativa. El CSV de este notebook es una salida exploratoria, no una partición para evaluación independiente.

## 9. Preparación para futuro modelado

Se divide la base original antes de calcular medianas, moda, categorías, escalas o PCA. Las estadísticas se aprenden con entrenamiento y se aplican sin reajustarlas a prueba. Se conserva la imputación por clase y sexo y se añade mediana global de respaldo para grupos sin edad observada.

Se usa tamaño de familia sin `sibsp` ni `parch`, para evitar su dependencia lineal exacta. Se conserva la clase como ordinal (supone separación numérica igual entre clases). El indicador de cubierta requiere revisar, en un uso real, si esa información existiría al predecir.

Esta partición demuestra el flujo; como el EDA ya examinó toda la muestra, no constituye una evaluación final independiente. Tampoco evita posibles dependencias familiares: faltan identificadores de familia o tiquete. Si se realiza validación cruzada, debe repetirse todo el ajuste dentro de cada pliegue.

In [18]:
predictores = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "deck"]
df_original = pd.read_csv(DATA / "titanic.csv")
X = df_original[predictores].copy()
y = df_original.survived.copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, stratify=y, random_state=42)
medianas = X_train.groupby(["pclass", "sex"], observed=True).age.median()
mediana_global = X_train.age.median()
moda_puerto = X_train.embarked.mode().iloc[0]

def preparar(datos):
    salida = datos.copy()
    salida["edad_faltante"] = salida.age.isna().astype(int)
    salida["tiene_cubierta"] = salida.deck.notna().astype(int)
    claves = pd.MultiIndex.from_frame(salida[["pclass", "sex"]])
    edad_grupo = pd.Series(medianas.reindex(claves).to_numpy(), index=salida.index)
    salida["age"] = salida.age.fillna(edad_grupo).fillna(mediana_global)
    salida["embarked"] = salida.embarked.fillna(moda_puerto)
    salida["sexo_mujer"] = salida.sex.eq("female").astype(int)
    salida["tamano_familia"] = salida.sibsp + salida.parch + 1
    salida["log_fare"] = np.log1p(salida.fare)
    return salida.drop(columns=["sex", "deck", "sibsp", "parch", "fare"])

train = preparar(X_train)
test = preparar(X_test)
nums = [c for c in train.columns if c != "embarked"]
codificador = ColumnTransformer([
    ("numericas", "passthrough", nums),
    ("puerto", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), ["embarked"])
])
preprocesador = make_pipeline(codificador, StandardScaler())
Z_train = preprocesador.fit_transform(train)
Z_test = preprocesador.transform(test)
columnas = codificador.get_feature_names_out()
pca_train = PCA(n_components=.90, svd_solver="full")
PC_train = pca_train.fit_transform(Z_train)
PC_test = pca_train.transform(Z_test)

# Verificación de integridad, alineación y ajuste exclusivo con entrenamiento.
assert X_train.index.intersection(X_test.index).empty
assert len(X_train) + len(X_test) == len(df_original)
assert train.index.equals(y_train.index) and test.index.equals(y_test.index)
assert np.isfinite(Z_train).all() and np.isfinite(Z_test).all()
assert Z_train.shape[1] == Z_test.shape[1] == len(columnas)
assert int(preprocesador[-1].n_samples_seen_) == len(X_train)
assert "survived" not in train and "alive" not in train
assert pca_train.explained_variance_ratio_.sum() >= .90

# Caso de borde: una clase/sexo sin mediana aprendida usa el respaldo global.
caso_nuevo = X_test.iloc[[0]].copy()
caso_nuevo["pclass"] = 4  # grupo sintético no visto; no pertenece a la base real
caso_nuevo["age"] = np.nan
assert preparar(caso_nuevo).age.iloc[0] == mediana_global

# Transformar valores extremos de prueba no debe cambiar lo aprendido.
media_antes = preprocesador[-1].mean_.copy()
medianas_antes = medianas.copy()
prueba_extrema = X_test.copy()
prueba_extrema["fare"] = 1_000_000
preprocesador.transform(preparar(prueba_extrema))
np.testing.assert_array_equal(preprocesador[-1].mean_, media_antes)
pd.testing.assert_series_equal(medianas, medianas_antes)

for nombre, valores, indices, objetivo in [
    ("train", Z_train, X_train.index, y_train),
    ("test", Z_test, X_test.index, y_test)]:
    exportado = pd.DataFrame(valores, index=indices, columns=columnas)
    exportado["survived"] = objetivo
    exportado.to_csv(OUT / f"titanic_{nombre}.csv", index_label="fila_origen")
print("Entrenamiento:", Z_train.shape, "Prueba:", Z_test.shape)
print("PCA ajustado con entrenamiento:", pca_train.n_components_, "componentes")
print("Varianza retenida en entrenamiento:", pca_train.explained_variance_ratio_.sum())

Entrenamiento: (712, 9) Prueba: (179, 9)
PCA ajustado con entrenamiento: 7 componentes
Varianza retenida en entrenamiento: 0.941025456243992


## 10. Archivos generados y límites

`data/titanic_procesado.csv` contiene el resultado exploratorio de 11 predictores escalados más `survived`. `reports/mejoras/titanic_train.csv` y `titanic_test.csv` contienen 712 y 179 filas, con 9 predictores, `survived` y `fila_origen`. Separar el objetivo y excluir el índice al modelar. Las estadísticas de esas particiones se aprenden solo con entrenamiento; aun así, el EDA previo sobre toda la muestra impide presentarlas como evaluación final independiente.

La comparación queda en `reports/mejoras/comparacion_pca.csv`. Ninguna visualización acredita rendimiento predictivo ni causalidad. En validación cruzada, reajustar todo el preprocesamiento dentro de cada pliegue.

Referencia: [scikit-learn sobre fuga de información](https://scikit-learn.org/stable/common_pitfalls.html).